## Cell 0 — Runtime probe and installation

In [ ]:
# =============================================================
# CELL 0 — Runtime probe + dependencies
# =============================================================
import os, sys, subprocess, platform, textwrap

def sh(cmd):
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, check=False)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed (exit {result.returncode}): {cmd}")

# ---- 0.1 Probe the machine we were actually given -------------
import multiprocessing
print("Python      :", sys.version.split()[0])
print("Platform    :", platform.platform())
print("CPU cores   :", multiprocessing.cpu_count(), "  <-- drives ORC runtime (see PRD 5.1)")
try:
    import psutil
    print("System RAM  : %.1f GB" % (psutil.virtual_memory().total / 1e9))
except ImportError:
    sh("cat /proc/meminfo | head -1")

try:
    import torch
    print("Torch       :", torch.__version__, "| CUDA:", torch.cuda.is_available())
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print("GPU         :", p.name, "| VRAM: %.1f GB" % (p.total_memory / 1e9))
except ImportError:
    raise RuntimeError("Torch is required to run this notebook. Install it "
                        "(e.g. 'pip install torch') and restart the runtime.")

# ---- 0.2 Install ---------------------------------------------
sh("pip install -q GraphRicciCurvature POT networkx>=3.4 pyarrow tqdm")
sh("pip install -q torch-geometric")
sh("pip install -q transformers sentence-transformers adapters")

# ---- 0.3 Drive for checkpoints -------------------------------
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_ROOT = '/content/drive/MyDrive/orca_gcn_caif'
except Exception:
    CACHE_ROOT = './orca_gcn_caif'          # local / CI fallback
os.makedirs(CACHE_ROOT, exist_ok=True)
print("Cache root  :", CACHE_ROOT)


## Cell 1 — Master configuration

In [ ]:
# =============================================================
# CELL 1 — CFG: the single source of every tunable value
# =============================================================
from dataclasses import dataclass, asdict, field
from typing import Optional, Tuple
import json, hashlib, os

@dataclass
class Config:
    # ---------- Phase 1: acquisition ----------
    OPENALEX_API_KEY: str = ""             # REQUIRED since 2026-02-13
    OPENALEX_BASE: str = "https://api.openalex.org"
    JOURNAL_IDS: Tuple[str, ...] = ("S137773608",)   # OpenAlex source IDs
    YEAR_START: int = 2022
    YEAR_END: int = 2023
    MAX_SEED_PAPERS: int = 5000       # PRD 1: target scale is 5,000-20,000 papers; 5000 is the floor
    MAX_CITERS: Optional[int] = 3000       # Bounded by default so Cell 3 stays
                                            # inside the PRD 5.3 budget for this
                                            # stage (8-25 min) instead of running
                                            # for hours. citing_works() has no cap
                                            # of its own otherwise (found via live
                                            # testing: 25 seeds pulled 26,600+
                                            # citer records and was still
                                            # climbing when interrupted). Raise or
                                            # set to None (unbounded, PRD default)
                                            # for a full production run.
    CITES_BATCH: int = 50                  # OR-filter IDs per request
    PER_PAGE: int = 200
    REQUEST_TIMEOUT: int = 30
    MAX_RETRIES: int = 5
    SELECT_FIELDS: str = ("id,doi,title,publication_year,publication_date,"
                          "primary_location,authorships,referenced_works,"
                          "topics,abstract_inverted_index,cited_by_count")

    # ---------- Phase 2a: ORC ----------
    ORC_ALPHA: float = 0.5                 # random-walk laziness
    ORC_METHOD: str = "OTDSinkhornMix"     # OTD | Sinkhorn | OTDSinkhornMix
    ORC_PROC_CAP: int = 12
    ORC_ISOLATED_VALUE: float = 0.0

    # ---------- Phase 2b: ORCA-GCN ----------
    TAU_CURV: float = 1.0
    GATE_MODE: str = "softplus"            # softplus | exp | sigmoid
    GCN_LAYERS: int = 2
    GCN_HIDDEN: int = 128
    GCN_DROPOUT: float = 0.5
    GCN_LR: float = 0.01
    GCN_EPOCHS: int = 200
    GCN_USE_TEXT_FEATURES: bool = True
    GCN_TEXT_PCA_DIM: int = 64

    # ---------- Phase 2c: temporal ----------
    LAMBDA_DECAY: Optional[float] = None   # None => MLE-fit per field (PRD 4.3)
    TEMPORAL_IMPOSSIBLE_VALUE: float = 0.0

    # ---------- Phase 2d: semantic ----------
    SEMANTIC_BACKEND: str = "specter2"     # specter2 | st-specter | scibert
    EMB_BATCH_SIZE: int = 32
    EMB_MAX_TOKENS: int = 512
    EMB_FP16: bool = True
    SEMANTIC_MISSING_VALUE: float = 0.5

    # ---------- Phase 2e: fusion ----------
    STANDARDIZE: str = "robust"            # robust | zscore | none
    FUSION_ORDER: Tuple[str, ...] = ("kappa", "h", "tau", "sigma")

    # ---------- Phase 3: trust head ----------
    LOGIT_CLAMP: float = 30.0              # PRD 2.3.3 — Test 11 depends on this
    HEAD_LR: float = 0.01
    HEAD_L2: float = 1e-4
    HEAD_EPOCHS: int = 300
    HEAD_POS_WEIGHT: Optional[float] = None   # None => auto from class balance
    WEAK_CARTEL_PCT: float = 0.99
    WEAK_AUTHOR_OVERLAP: float = 0.50
    WEAK_BULK_REF_MIN: int = 5
    WEAK_POS_MIN_CITED: int = 10

    # ---------- Phase 4: metric + validation ----------
    FIELD_PEER_K: int = 20
    NULL_MODE: str = "strict"              # fast | strict
    NULL_B_FAST: int = 500
    NULL_B_STRICT: int = 50
    NULL_SWAP_FACTOR: int = 10
    Z_THRESHOLD: float = 1.96
    RECAL_VAR_TRIGGER: float = 0.30
    RECAL_MAX_PASSES: int = 2

    # ---------- infrastructure ----------
    SEED: int = 42
    USE_CACHE: bool = True
    CACHE_ROOT: str = CACHE_ROOT
    LOG_LEVEL: str = "INFO"

CFG = Config()

def _resolve_api_key():
    """Colab Secrets first (never paste a key into a cell or chat), then
    the OPENALEX_API_KEY env var for local/CI runs. See README.md."""
    try:
        from google.colab import userdata
        key = userdata.get("OPENALEX_API_KEY")
        if key:
            return key
    except Exception:
        pass
    return os.environ.get("OPENALEX_API_KEY", "")

CFG.OPENALEX_API_KEY = _resolve_api_key()
_raw_cache_path = os.path.join(CFG.CACHE_ROOT, "raw_works.pkl")
assert CFG.OPENALEX_API_KEY or (CFG.USE_CACHE and os.path.exists(_raw_cache_path)), (
    "No OpenAlex API key found and no cached raw_works.pkl to fall back on. "
    "In Colab: Secrets panel (key icon, left "
    "sidebar) -> add OPENALEX_API_KEY -> enable notebook access. "
    "Locally: set the OPENALEX_API_KEY env var. Get a free key at "
    "openalex.org/settings/api (PRD 3.5)."
)

def cfg_hash(c) -> str:
    return hashlib.sha256(json.dumps(asdict(c), sort_keys=True,
                                     default=str).encode()).hexdigest()[:12]

CFG_HASH = cfg_hash(CFG)
print("Config hash:", CFG_HASH)
with open(os.path.join(CFG.CACHE_ROOT, f"cfg_{CFG_HASH}.json"), "w") as f:
    json.dump(asdict(CFG), f, indent=2, default=str)


## Cell 2 — Utilities: cache, retry, determinism

In [ ]:
# =============================================================
# CELL 2 — Infrastructure utilities
# =============================================================
import os, sys, gzip, pickle, random, logging, time, functools
import numpy as np, pandas as pd

# stream=sys.stdout (not the StreamHandler default of sys.stderr) + force=True:
# ipykernel swaps sys.stdout/stderr for its own per-cell-output stream, but a
# StreamHandler binds whatever `sys.stderr` object exists AT CONSTRUCTION TIME.
# If logging is configured before that swap settles (or basicConfig() no-ops on
# a rerun because handlers already exist), log.info() calls silently vanish
# instead of reaching the visible cell output -- print() still works because
# it re-reads sys.stdout on every call rather than caching a reference.
logging.basicConfig(level=getattr(logging, CFG.LOG_LEVEL),
                    format="%(asctime)s | %(levelname)-7s | %(message)s",
                    stream=sys.stdout, force=True)
log = logging.getLogger("caif")

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
    except ImportError:
        pass
set_seed(CFG.SEED)

# ---- checkpoint helpers: every expensive cell uses these ------
# pickle is used only for this notebook's own Drive cache (CFG.CACHE_ROOT):
# files this same run wrote and reads back, never an externally-sourced
# artifact. Not a deserialization-of-untrusted-input risk in this context.
def _p(name, ext): return os.path.join(CFG.CACHE_ROOT, f"{name}.{ext}")

def cache_exists(name, ext="pkl"): return CFG.USE_CACHE and os.path.exists(_p(name, ext))

def cache_save(name, obj, ext="pkl"):
    with open(_p(name, ext), "wb") as f: pickle.dump(obj, f, protocol=4)
    log.info("cached -> %s", _p(name, ext))

def cache_load(name, ext="pkl"):
    with open(_p(name, ext), "rb") as f: obj = pickle.load(f)
    log.info("loaded <- %s", _p(name, ext)); return obj

def stage(name, ext="pkl"):
    """Decorator: skip an expensive stage entirely if its artifact exists."""
    def deco(fn):
        @functools.wraps(fn)
        def wrap(*a, **kw):
            if cache_exists(name, ext): return cache_load(name, ext)
            t0 = time.time(); out = fn(*a, **kw)
            log.info("stage %-12s took %.1fs", name, time.time() - t0)
            cache_save(name, out, ext); return out
        return wrap
    return deco

def safe_finite(x, nan=0.0, posinf=1.0, neginf=-1.0):
    """Test 03: no NaN / Inf may ever enter the feature vector."""
    return np.nan_to_num(np.asarray(x, dtype=np.float64),
                         nan=nan, posinf=posinf, neginf=neginf)


## Cell 3 — Phase 1: OpenAlex acquisition

In [ ]:
# =============================================================
# CELL 3 — PHASE 1a: OpenAlex acquisition (quota-aware)
# =============================================================
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm.auto import tqdm

class OpenAlexClient:
    """Quota-aware client. Since 2026-02-13 an API key is REQUIRED
    ($0.10/day without, $1.00/day with a free key)."""

    def __init__(self, cfg):
        self.cfg = cfg
        self.s = requests.Session()
        self.s.headers.update({"User-Agent": "ORCA-GCN-CAIF/1.0 (academic research)"})
        self.s.mount("https://", HTTPAdapter(max_retries=Retry(
            total=cfg.MAX_RETRIES, backoff_factor=1.5,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=["GET"])))
        self.n_calls = 0
        self.last_usage = None

    def _get(self, path, params):
        params = dict(params); params["api_key"] = self.cfg.OPENALEX_API_KEY
        # urllib3's Retry (below) only covers bad status codes -- a connection
        # that drops mid-stream after a 200 already started (ChunkedEncodingError)
        # bypasses it entirely and would otherwise kill the whole acquisition.
        last_exc = None
        for attempt in range(self.cfg.MAX_RETRIES):
            try:
                r = self.s.get(f"{self.cfg.OPENALEX_BASE}/{path}", params=params,
                               timeout=self.cfg.REQUEST_TIMEOUT)
                break
            except (requests.exceptions.ChunkedEncodingError,
                    requests.exceptions.ConnectionError) as e:
                last_exc = e
                time.sleep(1.5 ** attempt)
        else:
            raise RuntimeError(
                f"OpenAlex connection kept dropping on GET {path} after "
                f"{self.cfg.MAX_RETRIES} attempts: {last_exc}"
            ) from last_exc
        self.n_calls += 1
        for h in ("x-usage-remaining", "x-ratelimit-remaining", "x-usage-cost"):
            if h in r.headers: self.last_usage = (h, r.headers[h])
        if r.status_code == 403:
            raise RuntimeError("OpenAlex 403 — API key missing/invalid, or daily budget spent.")
        try:
            r.raise_for_status()
        except requests.HTTPError as e:
            # Bare raise_for_status() loses which endpoint/call caused it once
            # it bubbles up through paginate() -> seed_works()/citing_works().
            raise RuntimeError(
                f"OpenAlex {r.status_code} on GET {path} (call #{self.n_calls}): {e}"
            ) from e
        return r.json()

    def paginate(self, path, params, max_results=None, desc="fetch"):
        """Cursor pagination. One call returns up to PER_PAGE results."""
        out, cursor = [], "*"
        pbar = tqdm(desc=desc, unit="rec")
        while cursor:
            p = dict(params); p.update({"per-page": self.cfg.PER_PAGE, "cursor": cursor,
                                        "select": self.cfg.SELECT_FIELDS})
            js = self._get(path, p)
            batch = js.get("results", [])
            out.extend(batch); pbar.update(len(batch))
            cursor = js.get("meta", {}).get("next_cursor")
            if not batch or (max_results and len(out) >= max_results): break
        pbar.close()
        return out[:max_results] if max_results else out

    def seed_works(self, journal_ids, y0, y1, max_papers):
        filt = (f"primary_location.source.id:{'|'.join(journal_ids)},"
                f"publication_year:{y0}-{y1}")
        return self.paginate("works", {"filter": filt},
                             max_results=max_papers, desc="seed works")

    def citing_works(self, work_ids, batch=50, max_citers=None):
        """CRITICAL: batch the reverse lookup. One call per paper would
        exceed the daily quota outright (PRD 2.3.1).

        max_citers caps the total across ALL chunks combined (None =
        unbounded, matching the original PRD design). Added after live
        testing showed a single journal's reverse-citation set can run to
        tens of thousands of records even from a small seed set -- without
        a cap this can blow the daily quota or run for a very long time."""
        ids = [w.split("/")[-1] for w in work_ids]
        out = []
        for i in tqdm(range(0, len(ids), batch), desc="citing works"):
            if max_citers is not None and len(out) >= max_citers:
                break
            chunk = "|".join(ids[i:i + batch])
            remaining = None if max_citers is None else max_citers - len(out)
            out.extend(self.paginate("works", {"filter": f"cites:{chunk}"},
                                     max_results=remaining, desc=""))
        return out[:max_citers] if max_citers is not None else out

@stage("raw_works")
def phase1_acquire(cfg):
    cl = OpenAlexClient(cfg)
    seed = cl.seed_works(cfg.JOURNAL_IDS, cfg.YEAR_START, cfg.YEAR_END,
                         cfg.MAX_SEED_PAPERS)
    log.info("seed works: %d", len(seed))
    citers = cl.citing_works([w["id"] for w in seed], cfg.CITES_BATCH, cfg.MAX_CITERS)
    log.info("citing works: %d | total API calls: %d | usage hdr: %s",
             len(citers), cl.n_calls, cl.last_usage)
    return {"seed": seed, "citers": citers,
            "seed_ids": {w["id"] for w in seed}}

RAW = phase1_acquire(CFG)


## Cell 4 — Phase 1: heterogeneous graph construction

In [ ]:
# =============================================================
# CELL 4 — PHASE 1b: typed heterogeneous graph
# =============================================================
import networkx as nx

def invert_abstract(inv_idx):
    """abstract_inverted_index -> plain text. Test 01: must tolerate None."""
    if not inv_idx: return None
    pos = {}
    for token, idxs in inv_idx.items():
        for i in idxs: pos[i] = token
    return " ".join(pos[i] for i in sorted(pos)) if pos else None

def _register_node(G, registry, node_id, **attrs):
    """Add a typed node to the graph and its lookup dict, once. Journal and
    author nodes both follow this same "create on first sight" pattern."""
    if node_id not in registry:
        registry[node_id] = attrs
        G.add_node(node_id, **attrs)


@stage("graph")
def phase1_build_graph(raw, cfg):
    G = nx.DiGraph()                     # heterogeneous, typed
    papers, authors, journals = {}, {}, {}
    all_works = {w["id"]: w for w in (raw["seed"] + raw["citers"])}

    for wid, w in all_works.items():
        loc = (w.get("primary_location") or {}).get("source") or {}
        jid = loc.get("id")
        rec = {
            "ntype": "paper",
            "year": w.get("publication_year"),
            "date": w.get("publication_date"),
            "journal": jid,
            "abstract": invert_abstract(w.get("abstract_inverted_index")),
            "title": w.get("title") or "",
            "topics": [t.get("id") for t in (w.get("topics") or []) if t.get("id")],
            "authors": [a.get("author", {}).get("id")
                        for a in (w.get("authorships") or [])],
            "institutions": [i.get("id") for a in (w.get("authorships") or [])
                             for i in (a.get("institutions") or [])],
            "cited_by_count": w.get("cited_by_count", 0),
            "is_seed": wid in raw["seed_ids"],
        }
        papers[wid] = rec
        G.add_node(wid, **rec)

        if jid:
            _register_node(G, journals, jid, ntype="journal", name=loc.get("display_name"))
            G.add_edge(wid, jid, etype="published_in")
        for aid in rec["authors"]:
            if not aid: continue
            _register_node(G, authors, aid, ntype="author")
            G.add_edge(wid, aid, etype="authored_by")

    # ---- citation edges: (citing) -[cites]-> (cited) ----------
    n_cite = 0
    for wid, w in all_works.items():
        for ref in (w.get("referenced_works") or []):
            if ref in all_works:                 # keep the graph closed
                G.add_edge(wid, ref, etype="cites"); n_cite += 1

    log.info("nodes: %d papers / %d authors / %d journals | cite edges: %d",
             len(papers), len(authors), len(journals), n_cite)
    return {"G": G, "papers": papers, "authors": authors, "journals": journals}

GRAPH = phase1_build_graph(RAW, CFG)

def citation_backbone(G, directed=True):
    """Paper->Paper subgraph. ORC needs the undirected projection."""
    E = [(u, v) for u, v, d in G.edges(data=True) if d.get("etype") == "cites"]
    H = nx.DiGraph() if directed else nx.Graph()
    H.add_nodes_from(n for n, d in G.nodes(data=True) if d.get("ntype") == "paper")
    H.add_edges_from(E)
    return H

DG  = citation_backbone(GRAPH["G"], directed=True)
UG  = citation_backbone(GRAPH["G"], directed=False)
log.info("backbone: %d nodes, %d directed edges, %d undirected",
         DG.number_of_nodes(), DG.number_of_edges(), UG.number_of_edges())


## Cell 5 — Phase 1: metadata extraction into four streams

In [ ]:
# =============================================================
# CELL 5 — PHASE 1c: four parallel streams
# =============================================================
from sklearn.preprocessing import MultiLabelBinarizer

@stage("streams")
def phase1_streams(graph, DG, cfg):
    P = graph["papers"]
    ids = list(DG.nodes())
    idx = {pid: i for i, pid in enumerate(ids)}

    # S_time
    s_time = pd.DataFrame([{"paper_id": p, "year": P[p]["year"],
                            "date": P[p]["date"]} for p in ids])

    # S_text  (None-tolerant: Test 01)
    s_text = {p: ((P[p]["title"] or "") + " [SEP] " + (P[p]["abstract"] or ""))
              for p in ids}
    n_missing = sum(1 for p in ids if not P[p]["abstract"])
    log.info("abstracts missing: %d / %d (%.1f%%)", n_missing, len(ids),
             100 * n_missing / max(len(ids), 1))

    # S_features : multi-hot topics
    mlb = MultiLabelBinarizer(sparse_output=False)
    X = mlb.fit_transform([P[p]["topics"] for p in ids]).astype("float32")
    if X.shape[1] == 0:                       # degenerate: no topics at all
        X = np.ones((len(ids), 1), dtype="float32")

    return {"ids": ids, "idx": idx, "s_time": s_time,
            "s_text": s_text, "X": X, "topic_vocab": list(mlb.classes_)}

STR = phase1_streams(GRAPH, DG, CFG)
log.info("feature matrix: %s", STR["X"].shape)


## Cell 6 — Phase 2: Ollivier-Ricci Curvature

In [ ]:
# =============================================================
# CELL 6 — PHASE 2a: ORC  (the CPU-bound stage; see PRD 5.1)
# =============================================================
import multiprocessing
from GraphRicciCurvature.OllivierRicci import OllivierRicci

def compute_orc(UG, cfg):
    """Per-connected-component ORC.

    ADAPTATION (PRD 3.2): GraphRicciCurvature relies on shortest-path
    distances; on a disconnected graph those are infinite. Test 08 requires
    that a two-component graph completes without an infinite loop, so we
    decompose first and merge after.
    """
    proc = min(multiprocessing.cpu_count(), cfg.ORC_PROC_CAP)
    kappa = {}
    comps = list(nx.connected_components(UG))
    log.info("ORC over %d component(s), proc=%d", len(comps), proc)

    for comp in tqdm(comps, desc="ORC components"):
        sub = UG.subgraph(comp).copy()
        if sub.number_of_edges() == 0:
            continue                                    # isolated node: no edges
        if sub.number_of_edges() < 2 or sub.number_of_nodes() < 3:
            for u, v in sub.edges():
                kappa[frozenset((u, v))] = cfg.ORC_ISOLATED_VALUE
            continue
        try:
            orc = OllivierRicci(sub, alpha=cfg.ORC_ALPHA, method=cfg.ORC_METHOD,
                                proc=proc, verbose="ERROR")
            orc.compute_ricci_curvature()
        except Exception as e:
            # GraphRicciCurvature errors (e.g. the Windows fork-context gap,
            # PRD 2.3.2) otherwise surface as a bare traceback with no link
            # back to which component/config triggered them.
            raise RuntimeError(
                f"ORC failed on a {sub.number_of_nodes()}-node / "
                f"{sub.number_of_edges()}-edge component "
                f"(alpha={cfg.ORC_ALPHA}, method={cfg.ORC_METHOD}, proc={proc}): {e}"
            ) from e
        for u, v, d in orc.G.edges(data=True):
            kappa[frozenset((u, v))] = float(d["ricciCurvature"])
    return kappa

@stage("kappa")
def phase2_orc(UG, cfg):
    return compute_orc(UG, cfg)

KAPPA = phase2_orc(UG, CFG)
kv = np.array(list(KAPPA.values()))
log.info("kappa: n=%d mean=%.4f std=%.4f min=%.4f max=%.4f | neg-share=%.1f%%",
         len(kv), kv.mean(), kv.std(), kv.min(), kv.max(), 100*(kv < 0).mean())

def kappa_of(u, v):
    return KAPPA.get(frozenset((u, v)), CFG.ORC_ISOLATED_VALUE)


## Cell 7 — Phase 2: semantic embeddings (GPU)

In [ ]:
# =============================================================
# CELL 7 — PHASE 2d: SPECTER2 embeddings  (the GPU stage)
# =============================================================
import torch

def load_semantic_model(cfg):
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    if cfg.SEMANTIC_BACKEND == "specter2":
        from transformers import AutoTokenizer
        from adapters import AutoAdapterModel
        tok = AutoTokenizer.from_pretrained("allenai/specter2_base")
        mdl = AutoAdapterModel.from_pretrained("allenai/specter2_base")
        mdl.load_adapter("allenai/specter2", source="hf",
                         load_as="proximity", set_active=True)
        mdl = mdl.to(dev).eval()
        if cfg.EMB_FP16 and dev == "cuda": mdl = mdl.half()
        return ("hf", tok, mdl, dev)
    if cfg.SEMANTIC_BACKEND == "st-specter":
        from sentence_transformers import SentenceTransformer
        return ("st", None,
                SentenceTransformer("sentence-transformers/allenai-specter",
                                    device=dev), dev)
    if cfg.SEMANTIC_BACKEND == "scibert":
        from transformers import AutoTokenizer, AutoModel  # ablation baseline
        tok = AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")
        mdl = AutoModel.from_pretrained("allenai/scibert_scivocab_uncased").to(dev).eval()
        return ("hf", tok, mdl, dev)
    raise ValueError(
        f"Unknown CFG.SEMANTIC_BACKEND={cfg.SEMANTIC_BACKEND!r}; "
        "expected 'specter2', 'st-specter', or 'scibert'."
    )

@stage("emb")
def phase2_embed(streams, cfg):
    kind, tok, mdl, dev = load_semantic_model(cfg)
    ids, texts = streams["ids"], [streams["s_text"][p] for p in streams["ids"]]

    if kind == "st":
        E = mdl.encode(texts, batch_size=cfg.EMB_BATCH_SIZE,
                       convert_to_numpy=True, show_progress_bar=True,
                       normalize_embeddings=True)
        return {"ids": ids, "E": E.astype("float32")}

    bs, out = cfg.EMB_BATCH_SIZE, []
    while True:
        try:
            out = []
            for i in tqdm(range(0, len(texts), bs), desc=f"embed (bs={bs})"):
                b = tok(texts[i:i + bs], padding=True, truncation=True,
                        max_length=cfg.EMB_MAX_TOKENS, return_tensors="pt").to(dev)
                with torch.no_grad():
                    h = mdl(**b).last_hidden_state[:, 0, :]     # CLS pooling
                out.append(h.float().cpu().numpy())
            break
        except torch.cuda.OutOfMemoryError:                     # PRD 2.4
            torch.cuda.empty_cache(); bs = max(bs // 2, 1)
            log.warning("CUDA OOM -> retrying with batch=%d", bs)
            if bs == 1: raise
    E = np.vstack(out)
    E /= (np.linalg.norm(E, axis=1, keepdims=True) + 1e-12)     # L2 normalize
    return {"ids": ids, "E": E.astype("float32")}

EMB = phase2_embed(STR, CFG)
EIDX = {p: i for i, p in enumerate(EMB["ids"])}
HAS_ABS = {p: bool(GRAPH["papers"][p]["abstract"]) for p in EMB["ids"]}

def semantic_sim(u, v, cfg=CFG):
    """Cosine similarity. Missing abstract -> neutral fill, never a silent 0."""
    if not (HAS_ABS.get(u) and HAS_ABS.get(v)):
        return cfg.SEMANTIC_MISSING_VALUE, True
    a, b = EMB["E"][EIDX[u]], EMB["E"][EIDX[v]]
    d = float(np.linalg.norm(a) * np.linalg.norm(b))
    return (0.0 if d == 0 else float(np.clip(np.dot(a, b) / d, -1.0, 1.0))), False


## Cell 8 — Phase 2: temporal encoder

In [ ]:
# =============================================================
# CELL 8 — PHASE 2c: temporal plausibility
# =============================================================
def fit_lambda(DG, papers, cfg):
    """MLE for exponential decay (PRD 4.3): lambda_hat = 1 / mean(dt)."""
    if cfg.LAMBDA_DECAY is not None:
        return float(cfg.LAMBDA_DECAY)
    dts = []
    for u, v in DG.edges():
        yu, yv = papers[u].get("year"), papers[v].get("year")
        if yu is None or yv is None: continue
        dt = yu - yv
        if dt >= 0: dts.append(dt)
    m = float(np.mean(dts)) if dts else 1.0
    lam = 1.0 / max(m, 1e-6)
    log.info("lambda fitted: mean dt = %.2f yr -> lambda = %.4f", m, lam)
    return lam

LAMBDA = fit_lambda(DG, GRAPH["papers"], CFG)

def temporal_score(u, v, papers=None, lam=None, cfg=CFG):
    """u cites v.  dt < 0 is chronologically impossible -> exactly 0.0 (Test 05)."""
    papers = papers if papers is not None else GRAPH["papers"]
    lam = LAMBDA if lam is None else lam
    yu, yv = papers[u].get("year"), papers[v].get("year")
    if yu is None or yv is None:
        return 0.5                                   # unknown -> neutral
    dt = yu - yv
    if dt < 0:
        return cfg.TEMPORAL_IMPOSSIBLE_VALUE          # hard block, exactly 0.0
    return float(np.exp(-lam * dt))


## Cell 9 — Phase 2: the ORCA-style GCN

In [ ]:
# =============================================================
# CELL 9 — PHASE 2b: ORCA-GCN (curvature-gated message passing)
# =============================================================
import torch.nn as nn, torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops

class ORCAConv(MessagePassing):
    """Curvature-gated graph convolution.

    ADAPTATION of CurvGN (Ye et al., ICLR 2020): CurvGN maps curvature
    through an MLP to per-channel weights for node classification. Here the
    gate is a single interpretable scalar, softplus(kappa/tau), so that
    TAU_CURV remains a calibratable knob and the gate stays strictly
    positive (an edge is never fully deleted, which would disconnect
    the graph).
    """
    def __init__(self, in_ch, out_ch, tau=1.0, mode="softplus"):
        super().__init__(aggr="add", flow="source_to_target")
        self.lin  = nn.Linear(in_ch, out_ch, bias=False)
        self.bias = nn.Parameter(torch.zeros(out_ch))
        self.tau, self.mode = tau, mode

    def gate(self, kappa):
        k = kappa / self.tau
        if self.mode == "softplus": return F.softplus(k).clamp(min=1e-12)
        if self.mode == "exp":      return torch.exp(k.clamp(max=10.0)).clamp(min=1e-12)
        return torch.sigmoid(k).clamp(min=1e-12)

    def forward(self, x, edge_index, kappa):
        x = self.lin(x)
        ei, kap = add_self_loops(edge_index, edge_attr=kappa,
                                 fill_value=1.0, num_nodes=x.size(0))
        a = self.gate(kap)                              # per-edge gate
        src, dst = ei[0], ei[1]
        deg = torch.zeros(x.size(0), device=x.device, dtype=a.dtype)
        deg.scatter_add_(0, dst, a)                     # normalize per receiver
        norm = a / deg.clamp(min=1e-12)[dst]
        return self.propagate(ei, x=x, norm=norm) + self.bias

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j


class ORCAGCN(nn.Module):
    def __init__(self, in_ch, hidden, layers=2, tau=1.0,
                 mode="softplus", dropout=0.5):
        super().__init__()
        self.convs = nn.ModuleList()
        d = in_ch
        for _ in range(layers):
            self.convs.append(ORCAConv(d, hidden, tau, mode)); d = hidden
        self.dropout = dropout

    def forward(self, x, edge_index, kappa):
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index, kappa)
            if i < len(self.convs) - 1:
                x = F.relu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)
        return x


def build_pyg_inputs(DG, streams, kappa_fn, emb, cfg):
    ids, idx = streams["ids"], streams["idx"]
    X = streams["X"]
    if cfg.GCN_USE_TEXT_FEATURES:
        from sklearn.decomposition import PCA
        k = min(cfg.GCN_TEXT_PCA_DIM, emb["E"].shape[1], emb["E"].shape[0])
        Xt = PCA(n_components=k, random_state=cfg.SEED).fit_transform(emb["E"])
        X = np.hstack([X, Xt.astype("float32")])
    edges = list(DG.edges())
    ei = torch.tensor([[idx[u] for u, v in edges],
                       [idx[v] for u, v in edges]], dtype=torch.long)
    kap = torch.tensor([kappa_fn(u, v) for u, v in edges], dtype=torch.float32)
    return torch.tensor(X, dtype=torch.float32), ei, kap, edges


@stage("gcn_h")
def phase2_gcn(DG, streams, emb, cfg):
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    X, ei, kap, edges = build_pyg_inputs(DG, streams, kappa_of, emb, cfg)
    X, ei, kap = X.to(dev), ei.to(dev), kap.to(dev)

    model = ORCAGCN(X.size(1), cfg.GCN_HIDDEN, cfg.GCN_LAYERS,
                    cfg.TAU_CURV, cfg.GATE_MODE, cfg.GCN_DROPOUT).to(dev)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.GCN_LR, weight_decay=5e-4)

    # Self-supervised link reconstruction: real edges vs. sampled non-edges.
    N, E = X.size(0), ei.size(1)
    for ep in range(cfg.GCN_EPOCHS):
        model.train(); opt.zero_grad()
        Z = model(X, ei, kap)
        Zn = F.normalize(Z, dim=1)
        pos = (Zn[ei[0]] * Zn[ei[1]]).sum(1)
        neg_src = torch.randint(0, N, (E,), device=dev)
        neg_dst = torch.randint(0, N, (E,), device=dev)
        neg = (Zn[neg_src] * Zn[neg_dst]).sum(1)
        loss = (F.binary_cross_entropy_with_logits(pos, torch.ones_like(pos)) +
                F.binary_cross_entropy_with_logits(neg, torch.zeros_like(neg)))
        loss.backward(); opt.step()
        if ep % 50 == 0: log.info("GCN epoch %3d loss %.4f", ep, loss.item())

    model.eval()
    with torch.no_grad():
        Zn = F.normalize(model(X, ei, kap), dim=1)
        h = (Zn[ei[0]] * Zn[ei[1]]).sum(1).cpu().numpy()
    return {"edges": edges, "h": h.astype("float32"), "node_emb": Zn.cpu().numpy()}

GCN_OUT = phase2_gcn(DG, STR, EMB, CFG)
NODE_EMB = GCN_OUT["node_emb"]

def h_of(u, v):
    """Cosine similarity between two nodes' learned representations -- the
    single source of truth for the GCN edge score, valid for ANY node pair
    (not just the original DG edges). Both Cell 10's fusion and Cell 12's
    null model call this so there is exactly one way to compute h(u, v)."""
    iu, iv = STR["idx"].get(u), STR["idx"].get(v)
    if iu is None or iv is None:
        return 0.0
    return float(np.dot(NODE_EMB[iu], NODE_EMB[iv]))


## Cell 10 — Phase 2: fusion and standardization

In [ ]:
# =============================================================
# CELL 10 — PHASE 2e: Multi-Modal Feature Vector
# =============================================================
from sklearn.preprocessing import RobustScaler, StandardScaler

@stage("phi", "pkl")
def phase2_fuse(DG, cfg):
    rows = []
    for u, v in tqdm(DG.edges(), total=DG.number_of_edges(), desc="fuse"):
        sig, missing = semantic_sim(u, v, cfg)
        rows.append({"src": u, "dst": v,
                     "kappa": kappa_of(u, v),
                     "h":     h_of(u, v),
                     "tau":   temporal_score(u, v, cfg=cfg),
                     "sigma": sig,
                     "sigma_missing": missing})
    df = pd.DataFrame(rows)

    cols = list(cfg.FUSION_ORDER)
    raw = df[cols].to_numpy(dtype=np.float64)
    n_bad = int((~np.isfinite(raw)).sum())
    if n_bad: log.warning("non-finite values repaired: %d (Test 03)", n_bad)
    raw = safe_finite(raw)                               # contract: no NaN/Inf
    df[cols] = raw

    scaler = {"robust": RobustScaler, "zscore": StandardScaler}.get(cfg.STANDARDIZE)
    if scaler is not None:
        sc = scaler().fit(raw)
        df[[c + "_z" for c in cols]] = sc.transform(raw)
    else:
        sc = None
        df[[c + "_z" for c in cols]] = raw

    assert df[[c + "_z" for c in cols]].shape[1] == len(cols), "Test 10: dim mismatch"
    return {"df": df, "scaler": sc, "cols": cols}

PHI = phase2_fuse(DG, CFG)
log.info("Phi assembled: %d edges x %d dims", len(PHI["df"]), len(PHI["cols"]))
print(PHI["df"][PHI["cols"]].describe().T[["mean", "std", "min", "max"]])


## Cell 11 — Phase 3: weak labels and the Citation Trust Head

In [ ]:
# =============================================================
# CELL 11 — PHASE 3: weak supervision + Citation Trust Head
# =============================================================
class CitationTrustHead(nn.Module):
    """w = sigmoid(W^T Phi + b), exactly as specified in the project book.

    LOGIT_CLAMP is not cosmetic: sigmoid(9999.) == 1.0 exactly in float64,
    which FAILS Test 11's open-interval requirement. Measured saturation
    point is a logit of 37; the default clamp of 30 yields
    0.9999999999999065 and 9.36e-14 -- both strictly inside (0,1) *in
    float64*. torch.sigmoid on a float32 tensor (PyTorch's default dtype)
    rounds sigmoid(30.0) to exactly 1.0, since 1 - 9.36e-14 is far below
    float32's ~1.2e-7 precision floor -- the clamp alone doesn't guarantee
    the open interval unless the final sigmoid itself runs in float64.
    """
    def __init__(self, d_in=4, logit_clamp=30.0):
        super().__init__()
        self.lin = nn.Linear(d_in, 1)
        self.logit_clamp = logit_clamp

    def logits(self, phi):
        return self.lin(phi).squeeze(-1).clamp(-self.logit_clamp, self.logit_clamp)

    def forward(self, phi):
        return torch.sigmoid(self.logits(phi).double())


def weak_labels(df, graph, cfg):
    """PRD 4.5 -- labels are derived ONLY from author / institution /
    journal / reciprocity metadata, all of which are ABSENT from Phi.
    Deriving them from Phi's own signals would make the head circular."""
    P = graph["papers"]
    lab = np.full(len(df), -1, dtype=np.int8)          # -1 = unlabelled

    # journal-pair reciprocity -> cartel signature
    from collections import Counter
    jpair = Counter()
    for u, v in zip(df.src, df.dst):
        ju, jv = P[u].get("journal"), P[v].get("journal")
        if ju and jv and ju != jv: jpair[(ju, jv)] += 1
    recip = {(a, b): min(jpair[(a, b)], jpair.get((b, a), 0))
             for (a, b) in jpair if (b, a) in jpair}
    thr = (np.quantile(list(recip.values()), cfg.WEAK_CARTEL_PCT)
           if recip else np.inf)

    for i, (u, v) in enumerate(zip(df.src, df.dst)):
        au, av = set(filter(None, P[u]["authors"])), set(filter(None, P[v]["authors"]))
        ju, jv = P[u].get("journal"), P[v].get("journal")
        overlap = len(au & av) / max(len(au | av), 1)
        if overlap >= cfg.WEAK_AUTHOR_OVERLAP:
            lab[i] = 0; continue                                   # self-citation loop
        if ju and jv and recip.get((ju, jv), 0) >= thr:
            lab[i] = 0; continue                                   # cartel
        iu = set(filter(None, P[u]["institutions"]))
        iv = set(filter(None, P[v]["institutions"]))
        if (not au & av) and (not iu & iv) and ju != jv \
           and P[v]["cited_by_count"] >= cfg.WEAK_POS_MIN_CITED:
            lab[i] = 1                                             # organic
    return lab


@stage("trust")
def phase3_audit(phi, graph, cfg, extra_sharpen=1.0):
    df, cols = phi["df"], [c + "_z" for c in phi["cols"]]
    y = weak_labels(df, graph, cfg)
    df["weak_label"] = y
    mask = y >= 0
    cov = mask.mean()
    log.info("weak-label coverage: %.2f%%  (pos=%d neg=%d)",
             100 * cov, int((y == 1).sum()), int((y == 0).sum()))
    if cov < 0.05:
        log.warning("coverage < 5%% -- report this as a limitation (PRD 4.5)")

    dev = "cuda" if torch.cuda.is_available() else "cpu"
    X = torch.tensor(df.loc[mask, cols].to_numpy(np.float32), device=dev)
    t = torch.tensor(y[mask].astype(np.float32), device=dev)

    head = CitationTrustHead(len(cols), cfg.LOGIT_CLAMP).to(dev)
    pw = (cfg.HEAD_POS_WEIGHT if cfg.HEAD_POS_WEIGHT
          else float((t == 0).sum() / max((t == 1).sum(), 1)))
    opt = torch.optim.Adam(head.parameters(), lr=cfg.HEAD_LR,
                           weight_decay=cfg.HEAD_L2)
    lossf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pw, device=dev))
    for ep in range(cfg.HEAD_EPOCHS):
        head.train(); opt.zero_grad()
        loss = lossf(head.logits(X), t); loss.backward(); opt.step()
        if ep % 100 == 0: log.info("head epoch %3d loss %.4f", ep, loss.item())

    head.eval()
    with torch.no_grad():
        Xa = torch.tensor(df[cols].to_numpy(np.float32), device=dev)
        w = torch.sigmoid(head.logits(Xa).double() * extra_sharpen).cpu().numpy()
    df["w"] = w
    W = head.lin.weight.detach().cpu().numpy().ravel()
    b = float(head.lin.bias.detach().cpu().numpy())
    log.info("learned W (standardized): %s | b=%.4f",
             dict(zip(phi["cols"], np.round(W, 4))), b)
    assert (w > 0).all() and (w < 1).all(), "Test 11: w must be inside (0,1)"
    return {"df": df, "W": W, "b": b, "coverage": float(cov), "head": head}

TRUST = phase3_audit(PHI, GRAPH, CFG)


## Cell 12 — Phase 4: CAIF, NCAIF, null model, Z gate

In [ ]:
# =============================================================
# CELL 12 — PHASE 4: impact calculation + statistical validation
# =============================================================
def caif_for_journal(df, graph, journal_id, cfg):
    """CFG.JOURNAL_IDS holds bare OpenAlex IDs (e.g. "S137773608") because
    that's what the API filter syntax expects, but Cell 4 stores each
    paper's journal as the full URL OpenAlex actually returns
    (loc.get("id") -> "https://openalex.org/S137773608"). Comparing them
    directly with == silently matches nothing, which makes CAIF always
    0.0 and every journal FLAGGED regardless of the real trust weights -
    normalize both sides to the bare ID before comparing."""
    P = graph["papers"]
    jid_bare = journal_id.rstrip("/").split("/")[-1]
    pubs = [p for p, r in P.items()
            if (r["journal"] or "").rstrip("/").split("/")[-1] == jid_bare
            and r["is_seed"]
            and cfg.YEAR_START <= (r["year"] or -1) <= cfg.YEAR_END]
    if not pubs:
        return {"CAIF": 0.0, "n_papers": 0, "n_citations": 0}
    pubset = set(pubs)
    inc = df[df.dst.isin(pubset)]
    return {"CAIF": float(inc.w.sum() / len(pubs)),
            "n_papers": len(pubs), "n_citations": int(len(inc))}


def rewire(G, seed, cfg):
    """Degree-preserving randomization.

    Test 16: nx raises NetworkXError('fewer than four nodes') on a triangle.
    The gate below is what makes the degenerate case terminate safely.
    """
    H = G.copy()
    if H.number_of_nodes() < 4 or H.number_of_edges() < 2:
        return H
    n = cfg.NULL_SWAP_FACTOR * H.number_of_edges()
    try:
        if H.is_directed():
            nx.directed_edge_swap(H, nswap=n, max_tries=20 * n, seed=seed)
        else:
            nx.double_edge_swap(H, nswap=n, max_tries=20 * n, seed=seed)
    except (nx.NetworkXError, nx.NetworkXAlgorithmError) as e:
        log.debug("rewire skipped: %s", e)
    return H


def null_distribution(DG, trust, graph, journal_id, cfg):
    """fast   : reuse kappa, recompute only the O(1) features (dev loop)
       strict : recompute ORC per permutation (reported result; expensive)"""
    B = cfg.NULL_B_FAST if cfg.NULL_MODE == "fast" else cfg.NULL_B_STRICT
    head, cols = trust["head"], [c + "_z" for c in PHI["cols"]]
    sc, dev = PHI["scaler"], next(head.parameters()).device
    out = []
    for b in tqdm(range(B), desc=f"null ({cfg.NULL_MODE}, B={B})"):
        Hd = rewire(DG, cfg.SEED + b, cfg)
        if cfg.NULL_MODE == "strict":
            kap_b = compute_orc(Hd.to_undirected(), cfg)
            kf = lambda u, v: kap_b.get(frozenset((u, v)), cfg.ORC_ISOLATED_VALUE)
        else:
            kf = kappa_of
        rows = []
        for u, v in Hd.edges():
            s, _ = semantic_sim(u, v, cfg)
            rows.append([kf(u, v), h_of(u, v),
                         temporal_score(u, v, cfg=cfg), s])
        A = safe_finite(np.array(rows))
        A = sc.transform(A) if sc is not None else A
        with torch.no_grad():
            wb = torch.sigmoid(head.logits(
                torch.tensor(A, dtype=torch.float32, device=dev)).double()).cpu().numpy()
        dfb = pd.DataFrame({"src": [u for u, _ in Hd.edges()],
                            "dst": [v for _, v in Hd.edges()], "w": wb})
        out.append(caif_for_journal(dfb, graph, journal_id, cfg)["CAIF"])
    return np.array(out)


def z_gate(score, null, cfg):
    mu, sd = float(np.mean(null)), float(np.std(null, ddof=1))
    if sd < 1e-12:
        log.warning("degenerate null (sigma~0): z forced to 0.0")
        return 0.0, False, mu, sd
    z = (score - mu) / sd
    return z, bool(z >= cfg.Z_THRESHOLD), mu, sd


FIELD_PEER_RESULTS = []   # append {"journal":..., "CAIF":...} from the peer run

def estimate_field_mean(peer_results, cfg):
    """Reduced-depth peer sample (PRD 2.3.4). Returns (mean, standard_error).

    Reported as an ESTIMATE of the field mean, never as the field mean.
    With no peer sample the normalizer is 1.0, so NCAIF == CAIF and the
    log says so loudly -- a missing normalizer must never masquerade as a
    real normalization.
    """
    vals = [r["CAIF"] for r in peer_results if np.isfinite(r.get("CAIF", np.nan))]
    if len(vals) < 2:
        log.warning("peer sample n=%d -> normalizer forced to 1.0 (NCAIF == CAIF)",
                    len(vals))
        return 1.0, 0.0
    mu = float(np.mean(vals))
    se = float(np.std(vals, ddof=1) / np.sqrt(len(vals)))
    log.info("field mean estimated from %d peers: %.4f (SE %.4f)", len(vals), mu, se)
    return (mu if mu > 1e-12 else 1.0), se


def phase4(trust, graph, DG, cfg, _pass=1):
    df = trust["df"]
    jid = cfg.JOURNAL_IDS[0]
    raw = caif_for_journal(df, graph, jid, cfg)
    mu_field, se_field = estimate_field_mean(FIELD_PEER_RESULTS, cfg)
    ncaif = raw["CAIF"] / max(mu_field, 1e-12)
    null = null_distribution(DG, trust, graph, jid, cfg)
    null_n = null / max(mu_field, 1e-12)
    z, sig, mu, sd = z_gate(ncaif, null_n, cfg)

    # ---- Test 18: adaptive re-calibration loopback -------------
    if (not sig) and float(df.w.std()) > cfg.RECAL_VAR_TRIGGER \
       and _pass < cfg.RECAL_MAX_PASSES:
        log.warning("low confidence (z=%.3f, std(w)=%.3f) -> Phase 3 pass %d",
                    z, df.w.std(), _pass + 1)
        retr = phase3_audit.__wrapped__(PHI, graph, cfg, extra_sharpen=1.5)
        return phase4(retr, graph, DG, cfg, _pass=_pass + 1)

    rep = {"journal": jid, **raw, "field_mean": mu_field, "field_se": se_field,
           "NCAIF": ncaif, "z": z, "null_mu": mu, "null_sigma": sd,
           "null_B": len(null), "null_mode": cfg.NULL_MODE,
           "decision": "PUBLISH" if sig else "FLAGGED",
           "passes": _pass, "W": trust["W"].tolist(), "b": trust["b"],
           "weak_label_coverage": trust["coverage"], "cfg_hash": CFG_HASH}
    log.info("RESULT %s", json.dumps(rep, indent=2))
    return rep

REPORT = phase4(TRUST, GRAPH, DG, CFG)
with open(os.path.join(CFG.CACHE_ROOT, "caif_report.json"), "w") as f:
    json.dump(REPORT, f, indent=2)


## Cell 13 — Test suite (all 18 Phase A tests)

In [ ]:
# =============================================================
# CELL 13 — TEST SUITE (all 18 Phase A tests)
# =============================================================
import traceback
RESULTS = []

def test(tid, name):
    def deco(fn):
        def run():
            try:
                fn(); RESULTS.append((tid, name, "PASS", ""))
                print(f"  [PASS] {tid} {name}")
            except Exception as e:
                status = "SKIP" if "cannot find context for 'fork'" in str(e) else "FAIL"
                RESULTS.append((tid, name, status, str(e)))
                print(f"  [{status}] {tid} {name}: {e}")
                traceback.print_exc()
        run.__name__ = f"test_{tid}"; return run
    return deco

# ---------- 01–03  Data ingestion ----------
@test("01", "API ingestion tolerates missing fields")
def t01():
    mock = {"id": "W1", "publication_year": 2022, "abstract_inverted_index": None,
            "authorships": [], "topics": None, "referenced_works": None,
            "primary_location": None}
    assert invert_abstract(mock["abstract_inverted_index"]) is None
    assert invert_abstract({}) is None
    assert invert_abstract({"deep": [1], "learning": [0]}) == "learning deep"

@test("02", "Graph edge construction builds typed triplets")
def t02():
    g = nx.DiGraph()
    g.add_node("W1", ntype="paper"); g.add_node("W2", ntype="paper")
    g.add_edge("W1", "W2", etype="cites")
    assert g.has_edge("W1", "W2") and not g.has_edge("W2", "W1")
    assert g["W1"]["W2"]["etype"] == "cites"
    assert citation_backbone(g).number_of_edges() == 1

@test("03", "No underflow / overflow / NaN reaches Phi")
def t03():
    x = np.array([np.nan, np.inf, -np.inf, 1e308 * 10, 1e-320])
    y = safe_finite(x)
    assert np.isfinite(y).all(), y                 # [VERIFIED] -> [0, 1, -1, 1, ~0]
    cols = PHI["cols"]
    assert np.isfinite(PHI["df"][cols].to_numpy()).all()

# ---------- 04–10  Feature extraction ----------
@test("04", "ORC: clique positive, bridge negative")
def t04():
    G = nx.Graph()
    for a in range(5):
        for b in range(a + 1, 5): G.add_edge(a, b)
    for a in range(5, 10):
        for b in range(a + 1, 10): G.add_edge(a, b)
    G.add_edge(4, 5)                                        # the bridge
    k = compute_orc(G, CFG)
    cl = [v for e, v in k.items()
          if max(e) < 5 or min(e) >= 5]
    br = k[frozenset((4, 5))]
    # [VERIFIED] clique mean = +0.5450, bridge = -0.6000  (alpha=0.5, OTD)
    assert all(v > 0 for v in cl), f"clique curvature not positive: {cl}"
    assert br < 0, f"bridge curvature not negative: {br}"

@test("05", "Temporal: impossible citation scores exactly 0")
def t05():
    P = {"A": {"year": 2019}, "B": {"year": 2021}}
    assert temporal_score("A", "B", P, 0.15) == 0.0        # [VERIFIED] exactly 0.0
    assert abs(temporal_score("B", "A", P, 0.15) - np.exp(-0.30)) < 1e-9  # 0.7408
    P2 = {"A": {"year": 2021}, "B": {"year": 2021}}
    assert temporal_score("A", "B", P2, 0.15) == 1.0

@test("06", "Semantic cosine: 1.0 identical, ~0 orthogonal")
def t06():
    v = np.random.RandomState(0).randn(768)
    cos = lambda a, b: float(np.dot(a, b) /
                             (np.linalg.norm(a) * np.linalg.norm(b)))
    assert abs(cos(v, v) - 1.0) < 1e-9                     # [VERIFIED] 1.0
    assert abs(cos(np.array([1., 0.]), np.array([0., 1.]))) < 1e-12   # [VERIFIED] 0.0

@test("07", "Zero-citation node: no division by zero")
def t07():
    G = nx.Graph(); G.add_node("iso")
    k = compute_orc(G, CFG)
    assert k == {}                                  # no edges -> no curvature
    assert kappa_of("iso", "nobody") == CFG.ORC_ISOLATED_VALUE
    P = {"iso": {"year": 2022}}
    assert 0.0 <= temporal_score("iso", "iso", P, 0.15) <= 1.0

@test("08", "Disconnected components: terminates, no infinite loop")
def t08():
    G = nx.Graph()
    G.add_edges_from([(0, 1), (1, 2), (0, 2)])            # component A
    G.add_edges_from([(10, 11), (11, 12), (10, 12)])      # component B
    assert nx.number_connected_components(G) == 2
    k = compute_orc(G, CFG)
    assert len(k) == 6, f"expected 6 edges, got {len(k)}"
    assert all(np.isfinite(v) for v in k.values())

@test("09", "Zeroed ORC attention suppresses only those edges")
def t09():
    dev = "cpu"
    x = torch.randn(10, 8)
    ei = torch.tensor([[0, 1, 4, 5], [1, 0, 5, 4]])
    kap  = torch.tensor([0.545, 0.545, -0.6, -0.6])
    conv = ORCAConv(8, 4, CFG.TAU_CURV, CFG.GATE_MODE).eval()
    with torch.no_grad():
        base = conv(x, ei, kap)
        supp = conv(x, ei, torch.tensor([0.545, 0.545, -1e9, -1e9]))
    # [VERIFIED] non-bridge nodes are bit-identical; bridge nodes change
    assert torch.allclose(base[1], supp[1], atol=1e-6), "non-bridge node moved"
    assert not torch.allclose(base[5], supp[5], atol=1e-6), "bridge not suppressed"

@test("10", "Fused dimension == sum of component dimensions")
def t10():
    d = [1, 1, 1, 1]
    phi = np.concatenate([np.zeros(x) for x in d])
    assert phi.shape[0] == sum(d) == 4                     # [VERIFIED]
    assert len(PHI["cols"]) == 4
    assert PHI["df"][[c + "_z" for c in PHI["cols"]]].shape[1] == 4

# ---------- 11–12  Auditing ----------
@test("11", "Sigmoid strictly inside (0,1) for magnitude 9999")
def t11():
    head = CitationTrustHead(4, CFG.LOGIT_CLAMP)
    x = torch.tensor([[9999.] * 4, [-9999.] * 4, [0.] * 4])
    w = head(x).detach().numpy()
    # [VERIFIED] naive sigmoid(9999) == 1.0 exactly and FAILS this assertion;
    # saturation measured at logit 37. Clamp of 30 -> 0.9999999999999065.
    assert (w > 0).all() and (w < 1).all(), f"outside open interval: {w}"
    assert float(np.max(w)) < 1.0 and float(np.min(w)) > 0.0

@test("12", "Trust Head converges on a repeated vector")
def t12():
    set_seed(0)
    head = CitationTrustHead(4, CFG.LOGIT_CLAMP)
    opt  = torch.optim.Adam(head.parameters(), lr=0.01)
    X = torch.tensor([[0.5, 0.5, 0.5, 0.5]] * 64)
    y = torch.ones(64)
    hist = []
    for _ in range(400):
        opt.zero_grad()
        loss = nn.BCEWithLogitsLoss()(head.logits(X), y)
        loss.backward(); opt.step(); hist.append(float(loss))
    assert hist[-1] < hist[0], "loss did not decrease"
    tail = np.array(hist[-50:])
    assert tail.std() < 1e-3, f"not asymptotically stable: std={tail.std():.2e}"
    assert np.isfinite([p.detach().numpy() for p in head.parameters()][0]).all()

# ---------- 13–14  Impact calculation ----------
@test("13", "CAIF summation matches manual arithmetic")
def t13():
    w = [0.94, 0.22, 0.5, 0.99, 0.01, 0.7, 0.3, 0.6, 0.8, 0.15]
    P = ["P%d" % i for i in range(5)]
    df = pd.DataFrame({"src": ["C%d" % i for i in range(10)],
                       "dst": [P[i % 5] for i in range(10)], "w": w})
    graph = {"papers": {p: {"journal": "J", "is_seed": True, "year": 2022}
                        for p in P}}
    r = caif_for_journal(df, graph, "J", CFG)
    assert abs(r["CAIF"] - sum(w) / 5) < 1e-12             # [VERIFIED] 1.042
    assert r["n_papers"] == 5 and r["n_citations"] == 10

@test("14", "Field normalization survives extreme scaling")
def t14():
    caif = 1.042
    for mu in (1e-12, 1e-6, 1.0, 1e6, 1e12):
        n = caif / max(mu, 1e-12)
        assert np.isfinite(n), mu                          # [VERIFIED] finite at 1e±12
    assert abs((caif / max(0.0, 1e-12)) - caif * 1e12) < 1e-3   # guard, no ZeroDivision

# ---------- 15–18  Validation and control loop ----------
@test("15", "Null rewiring preserves the degree sequence")
def t15():
    G = nx.gnp_random_graph(30, 0.15, seed=1, directed=True)
    ind = sorted(d for _, d in G.in_degree())
    outd = sorted(d for _, d in G.out_degree())
    H = rewire(G, 3, CFG)
    # [VERIFIED] both sequences preserved exactly across 10x|E| swaps
    assert sorted(d for _, d in H.in_degree()) == ind
    assert sorted(d for _, d in H.out_degree()) == outd
    assert H.number_of_edges() == G.number_of_edges()
    assert set(H.edges()) != set(G.edges()), "graph did not actually rewire"

@test("16", "Degenerate graph rewiring terminates safely")
def t16():
    tri = nx.cycle_graph(3)
    H = rewire(tri, 2, CFG)                    # nx raises without the guard
    # [VERIFIED] guard returns the graph unchanged, degree sequence intact
    assert sorted(d for _, d in H.degree()) == sorted(d for _, d in tri.degree())
    assert rewire(nx.Graph(), 0, CFG).number_of_nodes() == 0

@test("17", "Z-gate rejects a score sitting on the null mean")
def t17():
    null = np.random.RandomState(1).normal(1.0, 0.1, 500)
    z, sig, mu, sd = z_gate(float(np.mean(null)), null, CFG)
    assert abs(z) < 1e-9 and sig is False           # [VERIFIED] z = 0.000000
    z2, s2, _, _ = z_gate(float(np.mean(null)) + 3 * np.std(null, ddof=1),
                          null, CFG)
    assert s2 is True and abs(z2 - 3.0) < 1e-6      # [VERIFIED] z = 3.000
    z3, s3, _, _ = z_gate(1.0, np.ones(100), CFG)   # degenerate null
    assert z3 == 0.0 and s3 is False

@test("18", "Re-calibration loopback triggers once and terminates")
def t18():
    calls = {"n": 0}
    def fake(_pass=1, z=0.5, wstd=0.9):
        calls["n"] += 1
        if z < CFG.Z_THRESHOLD and wstd > CFG.RECAL_VAR_TRIGGER \
           and _pass < CFG.RECAL_MAX_PASSES:
            return fake(_pass + 1, z, wstd)
        return _pass
    assert fake() == CFG.RECAL_MAX_PASSES == 2
    assert calls["n"] == 2, "loop did not terminate at RECAL_MAX_PASSES"

for f in [t01, t02, t03, t04, t05, t06, t07, t08, t09,
          t10, t11, t12, t13, t14, t15, t16, t17, t18]:
    f()

rep = pd.DataFrame(RESULTS, columns=["id", "name", "status", "detail"])
print("\n%d/%d PASS" % ((rep.status == "PASS").sum(), len(rep)))
display(rep)
rep.to_json(os.path.join(CFG.CACHE_ROOT, "test_report.json"),
            orient="records", indent=2)


## Cell 14 — Calibration sweep

In [ ]:
# =============================================================
# CELL 14 — Calibration sweep (cheap: reuses cached kappa + embeddings)
# =============================================================
import itertools, copy

def sweep(grid, phi, graph, cfg):
    rows = []
    for combo in itertools.product(*grid.values()):
        c = copy.deepcopy(cfg)
        for k, v in zip(grid.keys(), combo): setattr(c, k, v)
        set_seed(c.SEED)
        tr = phase3_audit.__wrapped__(phi, graph, c)      # bypass the cache
        df = tr["df"]; m = df.weak_label >= 0
        from sklearn.metrics import average_precision_score
        auprc = (average_precision_score(df.loc[m, "weak_label"], df.loc[m, "w"])
                 if m.sum() > 10 else float("nan"))
        rows.append({**dict(zip(grid.keys(), combo)),
                     "auprc": auprc, "w_mean": float(df.w.mean()),
                     "w_std": float(df.w.std()),
                     **{f"W_{n}": float(v) for n, v in zip(phi["cols"], tr["W"])}})
        log.info("sweep %s -> AUPRC %.4f", dict(zip(grid.keys(), combo)), auprc)
    return pd.DataFrame(rows).sort_values("auprc", ascending=False)

GRID = {"TAU_CURV":  [0.5, 1.0, 2.0],
        "GATE_MODE": ["softplus", "sigmoid"],
        "HEAD_LR":   [0.003, 0.01, 0.03],
        "STANDARDIZE": ["robust", "zscore"]}
SWEEP = sweep(GRID, PHI, GRAPH, CFG)
SWEEP.to_csv(os.path.join(CFG.CACHE_ROOT, "sweep.csv"), index=False)
display(SWEEP.head(10))
